In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [20]:
import pdfplumber
import os

# Set this to either a directory containing PDFs or a single PDF file.
pdf_input = r'D:\CourseAssist\config\data'                        # directory OR

all_text = ""

if os.path.isdir(pdf_input):
    pdf_files = [f for f in os.listdir(pdf_input) if f.lower().endswith('.pdf')]
    if not pdf_files:
        raise FileNotFoundError(f"No PDF files found in directory {pdf_input!r}")
    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdf_input, pdf_file)
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                all_text += page.extract_text() or ""
                all_text += "\n"
elif os.path.isfile(pdf_input):
    with pdfplumber.open(pdf_input) as pdf:
        for page in pdf.pages:
            all_text += page.extract_text() or ""
            all_text += "\n"
else:
    raise FileNotFoundError(f"Provided path is neither a directory nor a file: {pdf_input!r}")

print(all_text[100])

c


In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.create_documents([all_text])

In [5]:
len(chunks)

364

In [6]:
chunks[10]

Document(metadata={}, page_content='holes. These lurk in the centers of galaxies, and are\nray burst, a tremendous blast of energy detectable across\nhuge: they can be millions or even billions of times the\nthe entire observable Universe. Gamma-ray bursts are in\nmass of the Sun! They probably formed at the same\na sense the birth cries of black holes.\ntime as their parent galaxies, but exactly how is not\nknown for sure. Perhaps each one started as a\na\notni gnillaf\nr e\nb t t\nl a\na M\nck')

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

In [8]:
vector_store.index_to_docstore_id

{0: '7522d295-8a72-4c28-9524-19ae754aa0c8',
 1: 'db08a96e-ae7e-499f-a458-3d5fad915420',
 2: '91eda611-fb1a-41df-8ea9-ddd7a711769d',
 3: '9a26d6e7-14e5-48d1-85d1-f010f773aecb',
 4: 'ac566805-8c8a-4cc6-878a-e43e827c77ac',
 5: '94006878-963a-43ac-aeb9-87d74c75ff8a',
 6: '06a85aad-9801-4c88-93bb-bb5220b10914',
 7: 'e63a8a90-2cf2-4faf-8e39-14f926a52f42',
 8: '710482dc-ec0c-4fef-a833-38522e83e280',
 9: '89824524-867e-42c6-af67-ecf3a809d9e3',
 10: 'db566540-7561-4e6e-9458-938c1ff033e8',
 11: '7f7f37aa-0b56-4afa-be85-ed43414048f8',
 12: '6be75a93-2abb-490f-9677-3742ccf58302',
 13: 'd7b508a1-9b71-4445-9db7-a2a7866c8cfb',
 14: 'c8c32077-cb2e-4990-927c-fff74af2dc07',
 15: '8a6cdd58-5d58-4a5c-a89b-d1f0064f7256',
 16: 'bbc61ea5-3e3b-4311-9a38-3c85a0824f13',
 17: '70e9638e-0a99-47f6-8a75-774a7f91a8a6',
 18: '89e6541c-0192-4f63-a269-cd88ce559b60',
 19: '1a651569-1398-4ad8-ab52-42c4f34131e9',
 20: 'c49a10b0-22ab-4619-a6e6-c0adf87961ca',
 21: 'c6af17a8-fa76-491f-a33d-8fd38e871f6f',
 22: '5aaa1fb7-bde2-

In [9]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [10]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F3E3502050>, search_kwargs={'k': 4})

In [11]:
retriever.invoke('What is newton first law of motion?')

[Document(id='ee053b78-02da-4aa5-9720-fb99ea6aff02', metadata={}, page_content='governs the motion of bodies. In this chapter, we turn to this\n4.4 Newton’s first law of motion\nbasic question.\n4.5 Newton’s second law of\nLet us first guess the answer based on our common\nmotion\nexperience. To move a football at rest, someone must kick it.\n4.6 Newton’s third law of motion\nTo throw a stone upwards, one has to give it an upward\n4.7 Conservation of momentum\npush. A breeze causes the branches of a tree to swing; a\n4.8 Equilibrium of a particle'),
 Document(id='b01f1d5e-f1d6-45fa-9a73-0507f3aba382', metadata={}, page_content='thus: “Everybody continues to be in its state of rest or of uniform motion in a straight line,\nunless compelled by some external force to act otherwise”. In simple terms, the First Law\nis “If external force on a body is zero, its acceleration is zero”.\n3. Momentum (p ) of a body is the product of its mass (m) and velocity (v) :\np = mv\n4. Newton’s second law

In [13]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [14]:
question          = "Explain Newton's First Law of Motion."
retrieved_docs    = retriever.invoke(question)

In [15]:
retrieved_docs

[Document(id='ee053b78-02da-4aa5-9720-fb99ea6aff02', metadata={}, page_content='governs the motion of bodies. In this chapter, we turn to this\n4.4 Newton’s first law of motion\nbasic question.\n4.5 Newton’s second law of\nLet us first guess the answer based on our common\nmotion\nexperience. To move a football at rest, someone must kick it.\n4.6 Newton’s third law of motion\nTo throw a stone upwards, one has to give it an upward\n4.7 Conservation of momentum\npush. A breeze causes the branches of a tree to swing; a\n4.8 Equilibrium of a particle'),
 Document(id='577731b1-8a6f-4c18-9548-2fd37e8bcc08', metadata={}, page_content='4.4 NEWTON’S FIRST LAW OF MOTION\nexternal force acting on it. Its acceleration,\nGalileo’s simple, but revolutionary ideas according to the first law, must be zero. If it is\ndethroned Aristotelian mechanics. A new in motion, it must continue to move with a\nmechanics had to be developed. This task was uniform velocity.\nReprint 2025-26\n52 PHYSICS\nMore often,

In [16]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'governs the motion of bodies. In this chapter, we turn to this\n4.4 Newton’s first law of motion\nbasic question.\n4.5 Newton’s second law of\nLet us first guess the answer based on our common\nmotion\nexperience. To move a football at rest, someone must kick it.\n4.6 Newton’s third law of motion\nTo throw a stone upwards, one has to give it an upward\n4.7 Conservation of momentum\npush. A breeze causes the branches of a tree to swing; a\n4.8 Equilibrium of a particle\n\n4.4 NEWTON’S FIRST LAW OF MOTION\nexternal force acting on it. Its acceleration,\nGalileo’s simple, but revolutionary ideas according to the first law, must be zero. If it is\ndethroned Aristotelian mechanics. A new in motion, it must continue to move with a\nmechanics had to be developed. This task was uniform velocity.\nReprint 2025-26\n52 PHYSICS\nMore often, however, we do not know all the The acceleration of the car cannot be accounted\n\nthus: “Everybody continues to be in its state of rest or of uniform motion 

In [17]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [18]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      governs the motion of bodies. In this chapter, we turn to this\n4.4 Newton’s first law of motion\nbasic question.\n4.5 Newton’s second law of\nLet us first guess the answer based on our common\nmotion\nexperience. To move a football at rest, someone must kick it.\n4.6 Newton’s third law of motion\nTo throw a stone upwards, one has to give it an upward\n4.7 Conservation of momentum\npush. A breeze causes the branches of a tree to swing; a\n4.8 Equilibrium of a particle\n\n4.4 NEWTON’S FIRST LAW OF MOTION\nexternal force acting on it. Its acceleration,\nGalileo’s simple, but revolutionary ideas according to the first law, must be zero. If it is\ndethroned Aristotelian mechanics. A new in motion, it must continue to move with a\nmechanics had to be developed. This task was uniform velocity.\nReprint 2

In [21]:
import os
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Read the key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

print("Loaded:", GOOGLE_API_KEY is not None)

Loaded: True


In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.2
)

In [25]:
answer = llm.invoke(final_prompt)
print(answer.content)

Newton's First Law of Motion states that "everybody continues to be in its state of rest or of uniform motion in a straight line, unless compelled by some external force to act otherwise". In simpler terms, if the external force on a body is zero, its acceleration is zero.
